In [2]:
import numpy as np
import pandas as pd
import math
import unicodedata
import re
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing e métricas
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

# Modelos Baselines e Lineares
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import (
    Ridge,
    ElasticNet,
    Lasso,
    SGDRegressor,
)

# Modelos baseados em Árvores (Ensembles)
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
    AdaBoostRegressor,
)

from sklearn.model_selection import train_test_split


# Modelos Avançados de Gradient Boosting e Redes Neurais
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
# from lightgbm import LGBMRegressor # <- Descomente apenas quando resolver a DLL no Windows
from sklearn.neural_network import MLPRegressor

In [6]:

from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from catboost import Pool

ARQUIVO_PARQUET = "dados/cnpq_com_capes.parquet"
TARGET = "valor_pago"

print("\nCarregando dataset...")

df = pd.read_parquet(ARQUIVO_PARQUET)

print("Shape original:", df.shape)

# opcional -> amostra para acelerar
df = df.sample(n=10000, random_state=42)
print("Shape após dropna:", df.shape)
q = df['valor_pago'].quantile(0.95)
df = df[df['valor_pago'] <= q]
df = df.drop(columns=['_record_number'])





Carregando dataset...
Shape original: (3274778, 24)
Shape após dropna: (10000, 24)


In [7]:
resultados = []

In [8]:
# target 

y = df["valor_pago"]

# features

colunas_remover = ["valor_pago"]
X = df.drop(columns=colunas_remover)

# colunas categóricas
cat_cols = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

for col in cat_cols:
    X[col] = X[col].fillna("nulo").astype(str)

# colunas numéricas

num_cols = [c for c in X.columns if c not in cat_cols]

for col in num_cols:
    X[col] = X[col].fillna(0)

# log target

y_log = np.log1p(y)

# split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_log,
    test_size=0.2,
    random_state=42
)

In [9]:
import optuna
from sklearn.metrics import mean_squared_error

# CatBoost + Optuna em paralelo: cada trial usa 1 thread e o Optuna distribui as trials entre os núcleos
# Isso costuma acelerar mais do que deixar cada trial disputar todos os cores ao mesmo tempo.

def objective_catBoost(trial):
    model = CatBoostRegressor(
        iterations=trial.suggest_int("iterations", 500, 3000),
        depth=trial.suggest_int("depth", 4, 10),
        learning_rate=trial.suggest_float(
            "learning_rate",
            0.01,
            0.2,
            log=True,
        ),
        l2_leaf_reg=trial.suggest_float(
            "l2_leaf_reg",
            1,
            20,
        ),
        random_strength=trial.suggest_float(
            "random_strength",
            0,
            10,
        ),
        bagging_temperature=trial.suggest_float(
            "bagging_temperature",
            0,
            5,
        ),
        loss_function="RMSE",
        verbose=False,
        random_seed=42,
        thread_count=1,
        allow_writing_files=False,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_cols,
    )

    pred = model.predict(X_test)

    return np.sqrt(mean_squared_error(y_test, pred))

study = optuna.create_study(direction="minimize")
study.optimize(objective_catBoost, n_trials=30, n_jobs=-1)

resultados.append({
    "Modelo": "CatBoost",
    "RMSE": study.best_value,
})

print(study.best_params)

[I 2026-06-01 19:11:27,094] A new study created in memory with name: no-name-d9a84ead-6b62-45a0-b18a-893b0887edcd
[I 2026-06-01 19:12:53,513] Trial 20 finished with value: 0.7139072927354981 and parameters: {'iterations': 589, 'depth': 5, 'learning_rate': 0.1261030199541817, 'l2_leaf_reg': 16.37068450073869, 'random_strength': 1.1272472244610299, 'bagging_temperature': 1.0348176392332435}. Best is trial 20 with value: 0.7139072927354981.
[I 2026-06-01 19:13:24,869] Trial 11 finished with value: 0.7236844057812359 and parameters: {'iterations': 1144, 'depth': 4, 'learning_rate': 0.011754363752025357, 'l2_leaf_reg': 13.781459803345745, 'random_strength': 6.406391565543039, 'bagging_temperature': 1.557974218613904}. Best is trial 20 with value: 0.7139072927354981.
[I 2026-06-01 19:14:09,816] Trial 2 finished with value: 0.7195464004291763 and parameters: {'iterations': 593, 'depth': 7, 'learning_rate': 0.19114533611835674, 'l2_leaf_reg': 19.881633813708156, 'random_strength': 2.6417776849

{'iterations': 1827, 'depth': 10, 'learning_rate': 0.013415596739833198, 'l2_leaf_reg': 3.735163126903448, 'random_strength': 8.071364358062361, 'bagging_temperature': 1.2087136152293965}


In [10]:
best_model = CatBoostRegressor(
    **study.best_params,
    loss_function="RMSE",
    random_seed=42,
    thread_count=-1,
    allow_writing_files=False,
)

best_model.fit(
    X_train,
    y_train,
    cat_features=cat_cols,
)

pred_log = best_model.predict(X_test)

pred_real = np.expm1(pred_log)
y_real = np.expm1(y_test)

0:	learn: 1.3236630	total: 414ms	remaining: 12m 35s
1:	learn: 1.3141633	total: 822ms	remaining: 12m 30s
2:	learn: 1.3040580	total: 1.23s	remaining: 12m 29s
3:	learn: 1.2943848	total: 1.64s	remaining: 12m 27s
4:	learn: 1.2849506	total: 1.99s	remaining: 12m 3s
5:	learn: 1.2754256	total: 2.41s	remaining: 12m 10s
6:	learn: 1.2660210	total: 2.77s	remaining: 11m 59s
7:	learn: 1.2568170	total: 3.15s	remaining: 11m 56s
8:	learn: 1.2478419	total: 3.54s	remaining: 11m 55s
9:	learn: 1.2393935	total: 3.58s	remaining: 10m 50s
10:	learn: 1.2303780	total: 3.95s	remaining: 10m 52s
11:	learn: 1.2220462	total: 3.99s	remaining: 10m 3s
12:	learn: 1.2133608	total: 4.24s	remaining: 9m 51s
13:	learn: 1.2053694	total: 4.56s	remaining: 9m 51s
14:	learn: 1.1972700	total: 4.9s	remaining: 9m 52s
15:	learn: 1.1899267	total: 5.32s	remaining: 10m 1s
16:	learn: 1.1821015	total: 5.68s	remaining: 10m 5s
17:	learn: 1.1762455	total: 5.71s	remaining: 9m 33s
18:	learn: 1.1688672	total: 5.82s	remaining: 9m 13s
19:	learn: 1.

In [ ]:
# métricas

mae = mean_absolute_error(y_real, pred_real)
rmse = root_mean_squared_error(y_real, pred_real)
r2_real = r2_score(y_real, pred_real)
r2_log = r2_score(y_test, pred_log)

print("\n===== RESULTADOS =====")

print(f"MAE  : {mae:,.2f}")
print(f"RMSE : {rmse:,.2f}")
print(f"R² real : {r2_real:.4f}")
print(f"R² log  : {r2_log:.4f}")


===== RESULTADOS =====
MAE  : 3,362.09
RMSE : 5,798.63
R² real : 0.4833
R² log  : 0.7016


In [3]:
# 2) Gráfico predição x real (escala real)

plt.figure(figsize=(8, 8))
sns.scatterplot(x=y_real, y=pred_real, alpha=0.5)
plt.plot(
    [y_real.min(), y_real.max()],
    [y_real.min(), y_real.max()],
    'r--'
)

plt.xlabel("Valor Real")
plt.ylabel("Valor Predito") 
plt.title("Predição vs Real (Escala Real)")
plt.xscale('log')
plt.yscale('log')
plt.grid(True, which="both", ls="--", linewidth=0.5)    
plt.show()


NameError: name 'y_real' is not defined

<Figure size 800x800 with 0 Axes>